In [1]:
import os
import warnings
warnings.filterwarnings("ignore")

from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.embeddings import HuggingFaceEmbeddings
from langchain_community.vectorstores import FAISS

from langchain_core.prompts import PromptTemplate
from langchain_classic.chains import RetrievalQA
from langchain_groq import ChatGroq


In [2]:
from dotenv import load_dotenv
load_dotenv()

http_proxy = os.getenv('http_proxy')
https_proxy = os.getenv('https_proxy')
HTTP_PROXY = os.getenv('HTTP_PROXY')
HTTPS_PROXY = os.getenv('HTTPS_PROXY')


os.environ["HF_TOKEN"] = os.getenv("HF_TOKEN")
os.environ["GROQ_API_KEY"] = os.getenv("GROQ_API_KEY")



In [3]:
def load_pdf(pdf_path):
    loader = PyPDFLoader(pdf_path)
    documents = loader.load()
    return documents


pdf_path = "/nmhs2/hari/work/UAV/Projects/rag_pipeline/pdfs/AI_Engineering.pdf"

docs = load_pdf(pdf_path)

print("Total pages:", len(docs))
print("Sample metadata:", docs[0].metadata)



Total pages: 535
Sample metadata: {'producer': 'Antenna House PDF Output Library 2.6.0 (Linux64)', 'creator': 'AH CSS Formatter V6.0 MR2 for Linux64 : 6.0.2.5372 (2012/05/16 18:26JST)', 'creationdate': '2024-12-04T13:39:11+00:00', 'author': 'Chip Huyen;', 'moddate': '2024-12-04T09:21:26-05:00', 'title': 'AI Engineering', 'trapped': '/False', 'ebx_publisher': "O'Reilly Media", 'source': '/nmhs2/hari/work/UAV/Projects/rag_pipeline/pdfs/AI_Engineering.pdf', 'total_pages': 535, 'page': 0, 'page_label': 'Cover'}


In [4]:
def chunk_documents(docs, chunk_size=1000, chunk_overlap=150):
    splitter = RecursiveCharacterTextSplitter(
        chunk_size=chunk_size,
        chunk_overlap=chunk_overlap
    )
    chunks = splitter.split_documents(docs)
    return chunks


documents = chunk_documents(docs)

print("Total chunks:", len(documents))

Total chunks: 1452


In [5]:
embeddings = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2"
)

# test
vec = embeddings.embed_query("What is AI?")
print("Vector size:", len(vec))


/tmp/ipykernel_834783/7328443.py:1: LangChainDeprecationWarning: The class `HuggingFaceEmbeddings` was deprecated in LangChain 0.2.2 and will be removed in 1.0. An updated version of the class exists in the `langchain-huggingface package and should be used instead. To use it run `pip install -U `langchain-huggingface` and import as `from `langchain_huggingface import HuggingFaceEmbeddings``.
  embeddings = HuggingFaceEmbeddings(
Loading weights: 100%|██████████| 103/103 [00:00<00:00, 580.91it/s, Materializing param=pooler.dense.weight]                             
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Vector size: 384


In [6]:
vector_store = FAISS.from_documents(documents, embeddings)

retriever = vector_store.as_retriever(
    search_type="similarity",
    search_kwargs={"k": 3}
)


In [7]:
llm = ChatGroq(
    model_name="openai/gpt-oss-120b", 
    temperature=0
)


In [8]:
custom_prompt = PromptTemplate(
    input_variables=["context", "question"],
    template="""
You are an AI research assistant.

Answer ONLY using the provided context.
If the answer is not found, say:
"I could not find the answer in the document."

Context:
{context}

Question:
{question}

Answer:
"""
)


In [9]:
qa_chain = RetrievalQA.from_chain_type(
    llm=llm,
    retriever=retriever,
    return_source_documents=True,
    chain_type="stuff",
    chain_type_kwargs={"prompt": custom_prompt}
)



In [16]:
query = input("\nAsk a question: ")
result = qa_chain.invoke({"query": query})

print("Question:\n", query)
print("Answer:\n", result["result"])


Question:
 what is RAG
Answer:
 RAG (Retrieval‑Augmented Generation) is a system that combines a retrieval component (which finds relevant documents or passages) with a generation component (the language model) so that the model can incorporate external information when producing its output. It is evaluated both on the quality of the retrieval step and on the quality of the final generated results.
